## Análisis de Raster de temperatura

In [11]:
import pandas as pd 
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent
from rasterio.windows import from_bounds
import geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import Transformer
import shapely
from rasterio.mask import mask
from shapely.geometry import mapping
from shapely.geometry import box
from rasterio.transform import xy
from collections import Counter
from pathlib import Path

In [17]:


parcelas_gdf = gpd.read_file('../../DataSet/DATASET_RETO_AGRO_2026/Parcelas_Reto_AGC_CONJUNTO_70_30/Parcelas_Reto_AGC_CONJUNTO.shp')

parcelas_gdf.head()

,Cultivo,ID_POLIGON,área_ha,Municipio,Estado,CONJUNTO,geometry
0,Cebada,AGC_196,8.57,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.57739 19.65802 0, -98.57751 19..."
1,Cebada,AGC_044,0.68,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.58655 19.62943 0, -98.5862 19...."
2,Cebada,AGC_020,8.34,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.58494 19.6278 0, -98.58656 19...."
3,CEBADA,AGC_174,6.89,Cuautepec de Hinojosa,Hidalgo,PREDICCION,"POLYGON Z ((-98.30604 19.9 0, -98.30486 19.899..."
4,CEBADA,AGC_083,2.87,Cuautepec de Hinojosa,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.29383 19.8882 0, -98.29378 19...."


In [18]:
print(f'Shape: {parcelas_gdf.shape}')
print(f'CRS: {parcelas_gdf.crs}')
print(f'Columns: {parcelas_gdf.columns}')

Shape: (197, 7)
CRS: EPSG:4326
Columns: Index(['Cultivo', 'ID_POLIGON', 'área_ha', 'Municipio', 'Estado', 'CONJUNTO',
       'geometry'],
      dtype='str')


In [4]:
tmax_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS/Reto_AgroCebada_Temperatura_2022_2025/Tmax/2025/Tmax_2025_01.tif'

print('Tmax:')

with rasterio.open(tmax_path) as src_tmax:
    print(f'CSR: {src_tmax.crs}')
    print(f'Dimensiones: {src_tmax.width} x {src_tmax.height}')
    print(f'Bandas: {src_tmax.count}')
    print(f'Resolución: {src_tmax.res}')
    print(f'Bounds: {src_tmax.bounds}')
    print(f'No Data: {src_tmax.nodata}')
    print(f'Data type: {src_tmax.dtypes[0]}')

Tmax:
CSR: EPSG:4326
Dimensiones: 65 x 72
Bandas: 1
Resolución: (0.05000000074505806, 0.05000000074505806)
Bounds: BoundingBox(left=-99.94999880716205, bottom=17.799999222159386, right=-96.69999875873327, top=21.399999275803566)
No Data: -9999.0
Data type: float32


In [7]:
tmin_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS/Reto_AgroCebada_Temperatura_2022_2025/Tmin/2025/Tmin_2025_01.tif'

print('Tmin')

with rasterio.open(tmin_path) as src_tmin:
    print(f'CSR: {src_tmin.crs}')
    print(f'Dimensiones: {src_tmin.width} x {src_tmin.height}')
    print(f'Bandas: {src_tmin.count}')
    print(f'Resolución: {src_tmin.res}')
    print(f'Bounds: {src_tmin.bounds}')
    print(f'No Data: {src_tmin.nodata}')
    print(f'Data type: {src_tmin.dtypes[0]}')

Tmin
CSR: EPSG:4326
Dimensiones: 65 x 72
Bandas: 1
Resolución: (0.05000000074505806, 0.05000000074505806)
Bounds: BoundingBox(left=-99.94999880716205, bottom=17.799999222159386, right=-96.69999875873327, top=21.399999275803566)
No Data: -9999.0
Data type: float32


In [13]:
temp_dir = Path('../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS/Reto_AgroCebada_Temperatura_2022_2025/')

temp_files = sorted(temp_dir.rglob('*.tif'))

print(f'Total de archivos tif: {len(temp_files)}')

for file in temp_files[:10]:
    print(file.name)

Total de archivos tif: 96
Tmax_2022_01.tif
Tmax_2022_02.tif
Tmax_2022_03.tif
Tmax_2022_04.tif
Tmax_2022_05.tif
Tmax_2022_06.tif
Tmax_2022_07.tif
Tmax_2022_08.tif
Tmax_2022_09.tif
Tmax_2022_10.tif


In [14]:
conteo_variables = Counter(
    ruta.name.split("_")[0]
    for ruta in temp_files
)

conteo_años = Counter(
    ruta.name.split("_")[1]
    for ruta in temp_files
)

print("Archivos por variable:", conteo_variables)
print("Archivos por año:", conteo_años)

# Revisar algunos archivos Tmin
for ruta in temp_files:
    if ruta.name.startswith("Tmin_"):
        print(ruta)
        break


Archivos por variable: Counter({'Tmax': 48, 'Tmin': 48})
Archivos por año: Counter({'2022': 24, '2023': 24, '2024': 24, '2025': 24})
..\..\DataSet\DATASET_RETO_AGRO_2026\Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS\Reto_AgroCebada_Temperatura_2022_2025\Tmin\2022\Tmin_2022_01.tif


In [15]:
def get_data_from_raster(raster_path, parcela, variable_name):
    with rasterio.open(raster_path) as src:
        parcela_geom = parcela.to_crs(src.crs)
        parcela_geom = parcela_geom.geometry.iloc[0]
        
        #Recortamos los pixeles del raster que intersacatn con la parcela
        values, transform = mask(src, [mapping(parcela_geom)], crop=True, filled=False, all_touched=True)
        
        #Objeto para transformar de coordendas a metros
        transfomer = Transformer.from_crs(src.crs, 'EPSG:6372', always_xy=True)
        
        #Parcela en metros
        parcela_m = parcela.to_crs('EPSG:6372')
        geometry_m = parcela_m.geometry.iloc[0]
        
        area_total = geometry_m.area
        area_valida = 0
        suma_ponderada = 0
        pixeles_validos = 0
        
        for i in range(values.shape[1]):
            for j in range(values.shape[2]):
                
                #Ignoramos los pixeles que no tienen datos
                if  np.ma.getmaskarray(values)[0, i, j]:
                    continue
                
                valor = float(values[0, i, j])

                if not np.isfinite(valor):
                    continue

                # Límites del píxel en el CRS del raster
                x1, y1 = xy(transform, i, j, offset="ul")

                x2, y2 = xy(transform, i, j, offset="lr")

                pixel_geo = box(x1, y2, x2, y1)

                # Transformar el píxel a coordenadas métricas
                pixel_m = shapely.ops.transform(transfomer.transform,pixel_geo)

                # Intersección real con la parcela
                interseccion = pixel_m.intersection(geometry_m)
                area_interseccion = interseccion.area

                if area_interseccion <= 0:
                    continue

                area_valida += area_interseccion
                suma_ponderada += valor * area_interseccion
                pixeles_validos += 1

        media_ponderada = (
            suma_ponderada / area_valida
            if area_valida > 0
            else np.nan
        )

        return {
            "area_total_m2": area_total,
            "area_valida_m2": area_valida,
            "cobertura_pct": 100 * area_valida / area_total,
            f"{variable_name}_media": media_ponderada,
            "pixeles_validos": pixeles_validos
        }

In [19]:

ruta_tmax = next(
    ruta for ruta in temp_files
    if ruta.name == "Tmax_2025_01.tif"
)

ruta_tmin = next(
    ruta for ruta in temp_files
    if ruta.name == "Tmin_2025_01.tif"
)

parcela_prueba = parcelas_gdf[
    parcelas_gdf["ID_POLIGON"] == "AGC_001"
]

resultado_tmax = get_data_from_raster(
    ruta_tmax, parcela_prueba, "temperatura_maxima"
)

resultado_tmin = get_data_from_raster(
    ruta_tmin, parcela_prueba, "temperatura_minima"
)

print("Tmax:", resultado_tmax)
print("Tmin:", resultado_tmin)


Tmax: {'area_total_m2': 75685.78283383907, 'area_valida_m2': 75685.78283383907, 'cobertura_pct': 100.0, 'temperatura_maxima_media': 20.859745025634766, 'pixeles_validos': 1}
Tmin: {'area_total_m2': 75685.78283383907, 'area_valida_m2': 75685.78283383907, 'cobertura_pct': 100.0, 'temperatura_minima_media': 6.540424346923828, 'pixeles_validos': 1}
